## 2) Convert Coco to ODVG

In [ ]:
import re

# Define the file path
file_path = '/GroundingDINO-Med-main/OpenGroundingDino/tools/coco2odvg.py'

# Define the new values according to the dataset
new_id_map = '{0: 0, 1:1}'
#new_ori_map = '{"0": "vertebra"}'
#new_ori_map = '{"0": "entrance"}'
#new_ori_map = '{"0": "fire", "1":"smoke"}'
new_ori_map = '{"0": "door symbol", "1": "window symbol", "2": "zone"}'
#new_ori_map = '{"0": "jellyfish", "1": "penguin", "2": "shark", "3": "starfish", "4": "stingray"}'
#new_ori_map = '{"0": "human head"}'

# Read the content of the file
with open(file_path, 'r') as file:
    content = file.read()

# Replace the id_map value using regex
content = re.sub(r'id_map\s*=\s*\{[^\}]*\}', f'id_map = {new_id_map}', content)

# Replace the ori_map value using regex
content = re.sub(r'ori_map\s*=\s*\{[^\}]*\}', f'ori_map = {new_ori_map}', content)

# Write the updated content back to the file
with open(file_path, 'w') as file:
    file.write(content)

print(f"Updated {file_path} successfully.")

Updated /GroundingDINO-Med-main/OpenGroundingDino/tools/coco2odvg.py successfully.


## 3) Convert COCO JSON to trainable parameter for grounding dino

In [ ]:
#!pip install jsonlines

In [13]:
# change path of input file to your input Coco json file

!python /GroundingDINO-Med-main/OpenGroundingDino/tools/coco2odvg.py --input "/GroundingDINO-Med-main/crowd_data/sample10/merged/train/_annotations.coco.json"  --output "/GroundingDINO-Med-main/input_params/train.jsonl"

loading annotations into memory...
Done (t=0.04s)
creating index...
index created!
  == dump meta ...
  == done.



100%|██████████| 10/10 [00:00<00:00, 346.47it/s]


In [ ]:
import json

# Define the content for the JSON file
#content = {"0": "jellyfish", "1": "penguin", "2": "shark", "3": "starfish", "4": "stingray"}
content = {"0": "door symbol", "1": "window symbol", "2": "zone"}
#content = {"0": "entrance"}
#content = {"0": "fire", "1":"smoke"}
#content = {"0": "vertebra"}

# Define the file path
file_path = '/GroundingDINO-Med-main/input_params/label.json'

# Write the content to the JSON file
with open(file_path, 'w') as file:
    json.dump(content, file)

print(f"File '{file_path}' created successfully.")

File '/GroundingDINO-Med-main/input_params/label.json' created successfully.


In [ ]:
#change the paths according to file locations
import json

# Define the data
data = {
    "train": [
        {
            "root": "/GroundingDINO-Med-main/floor_plan/sample10/merged/train",#Train images
            "anno": "/GroundingDINO-Med-main/input_params/train.jsonl",#Odvg jsonl file
            "label_map": "/GroundingDINO-Med-main/input_params/label.json",# label.json file
            "dataset_mode": "odvg"
        }
    ],
    "val": [
        {
            "root": "/GroundingDINO-Med-main/floor_plan/sample/merged/valid",#Test Images
            "anno": "/GroundingDINO-Med-main/floor_plan/sample/merged/valid/_annotations.coco.json",#Test data Annotation file in COCO
            "label_map": None,
            "dataset_mode": "coco"
        }
    ]
}

file_path = '/GroundingDINO-Med-main/OpenGroundingDino/config/datasets_mixed_odvg.json'

with open(file_path, 'w') as file:
    json.dump(data, file, indent=2)

print(f"Data has been written to {file_path}")

Data has been written to /GroundingDINO-Med-main/OpenGroundingDino/config/datasets_mixed_odvg.json


In [ ]:
import re

def modify_file(file_path):
    #label_list_content = 'label_list = ["jellyfish", "penguin", "shark", "starfish", "stingray"]\n'
    label_list_content = 'label_list = ["door symbol", "window symbol", "zone"]\n'
    #label_list_content = 'label_list = ["entrance"]\n'
    #label_list_content = 'label_list = ["fire", "smoke"]\n'
    #label_list_content = 'label_list = ["vertebra"]\n'
    # Read the entire content of the file
    with open(file_path, 'r') as file:
        content = file.read()

    # Replace use_coco_eval =TRUE with use_coco_eval =FALSE using regex
    content = re.sub(r'use_coco_eval\s*=\s*True', 'use_coco_eval = False', content)

    # Insert label_list after use_coco_eval = FALSE using regex
    content = re.sub(r'use_coco_eval\s*=\s*False', r'use_coco_eval = False\n\n' + label_list_content, content, count=1, flags=re.MULTILINE)

    # Write the modified content back to the file
    with open(file_path, 'w') as file:
        file.write(content)

# Paths to the files
cfg_coco_path = '/GroundingDINO-Med-main/OpenGroundingDino/config/cfg_coco.py'
cfg_odvg_path = '/GroundingDINO-Med-main/OpenGroundingDino/config/cfg_odvg.py'

# Modify both files
modify_file(cfg_coco_path)
modify_file(cfg_odvg_path)

print("Updated use_coco_eval to FALSE and added label_list using regex in both files.")

Updated use_coco_eval to FALSE and added label_list using regex in both files.


## 3) Train the model 

In [27]:
import torch

torch.cuda.empty_cache()

In [ ]:
!python "C:/GroundingDINO-Med-main/OpenGroundingDino/main.py" --config_file "C:/GroundingDINO-Med-main/OpenGroundingDino/config/cfg_odvg.py" --datasets "C:/GroundingDINO-Med-main/OpenGroundingDino/config/datasets_mixed_odvg.json" --output_dir "C:/GroundingDINO-Med-main/output/sample10" --pretrain_model_path "C:/GroundingDINO-Med-main/groundingdino_swint_ogc.pth"